# 網頁更新監控爬蟲

這個爬蟲會定期檢查 https://www.cs.ccu.edu.tw/~damon/secure/course-wk.html 是否有更新。

## 最快實作方法：
1. 使用 `requests` 處理登入和抓取網頁
2. 使用 `hashlib` 計算內容的哈希值來偵測變化
3. 儲存上次的哈希值，與新的哈希值比較
4. 使用 Linux 的 `cron` 或 Python 的 `schedule` 庫來定期執行

In [ ]:
# 安裝必要的套件（如果還沒安裝）
!pip install requests beautifulsoup4 schedule -q

In [ ]:
import requests
from bs4 import BeautifulSoup
import hashlib
import json
import os
from datetime import datetime

class WebsiteMonitor:
    def __init__(self, username, password, url):
        self.username = username
        self.password = password
        self.url = url
        self.session = requests.Session()
        self.hash_file = "page_hash.json"
        
    def login(self):
        """登入網站（需根據實際登入頁面調整）"""
        # 這裡需要找到實際的登入 URL 和表單欄位
        # 先嘗試訪問目標頁面，如果需要登入會被重定向
        
        response = self.session.get(self.url)
        
        # 如果使用 HTTP Basic Authentication
        if response.status_code == 401:
            response = self.session.get(
                self.url,
                auth=(self.username, self.password)
            )
            return response.status_code == 200
        
        # 如果是表單登入，需要找到登入頁面的 URL 和表單欄位名稱
        # 以下是一般的表單登入範例：
        # login_url = "https://www.cs.ccu.edu.tw/login"  # 實際登入網址
        # login_data = {
        #     'username': self.username,
        #     'password': self.password
        # }
        # response = self.session.post(login_url, data=login_data)
        
        return True
    
    def get_page_content(self):
        """取得網頁內容"""
        try:
            response = self.session.get(self.url)
            if response.status_code == 200:
                return response.text
            else:
                print(f"取得網頁失敗，狀態碼: {response.status_code}")
                return None
        except Exception as e:
            print(f"取得網頁時發生錯誤: {e}")
            return None
    
    def calculate_hash(self, content):
        """計算內容的 SHA256 哈希值"""
        return hashlib.sha256(content.encode('utf-8')).hexdigest()
    
    def load_previous_hash(self):
        """讀取上次儲存的哈希值"""
        if os.path.exists(self.hash_file):
            with open(self.hash_file, 'r') as f:
                data = json.load(f)
                return data.get('hash'), data.get('last_check')
        return None, None
    
    def save_hash(self, hash_value):
        """儲存新的哈希值"""
        data = {
            'hash': hash_value,
            'last_check': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        with open(self.hash_file, 'w') as f:
            json.dump(data, f, indent=2)
    
    def check_updates(self):
        """檢查網頁是否有更新"""
        print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] 開始檢查網頁...")
        
        # 登入
        if not self.login():
            print("登入失敗！")
            return
        
        # 取得網頁內容
        content = self.get_page_content()
        if not content:
            return
        
        # 計算新的哈希值
        new_hash = self.calculate_hash(content)
        
        # 讀取舊的哈希值
        old_hash, last_check = self.load_previous_hash()
        
        if old_hash is None:
            print("首次檢查，儲存初始哈希值。")
            self.save_hash(new_hash)
        elif old_hash != new_hash:
            print("⚠️ 網頁有更新！")
            print(f"上次檢查: {last_check}")
            print(f"舊哈希值: {old_hash[:16]}...")
            print(f"新哈希值: {new_hash[:16]}...")
            self.save_hash(new_hash)
            
            # 可以在這裡加入通知功能（如發送郵件、Line 通知等）
            # 也可以儲存完整的網頁內容以便比對差異
            self.save_page_content(content)
        else:
            print("✓ 網頁沒有變更。")
            print(f"上次檢查: {last_check}")
    
    def save_page_content(self, content):
        """儲存網頁內容快照"""
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"page_snapshot_{timestamp}.html"
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(content)
        print(f"已儲存網頁快照: {filename}")

# 使用範例
# 請替換成你的實際帳號密碼
USERNAME = "your_username"
PASSWORD = "your_password"
URL = "https://www.cs.ccu.edu.tw/~damon/secure/course-wk.html"

monitor = WebsiteMonitor(USERNAME, PASSWORD, URL)
monitor.check_updates()

## 方法一：使用 schedule 庫（在 Python 中執行）

適合在本機長時間運行或在 Jupyter Notebook 中測試。

In [ ]:
import schedule
import time

# 設定每天特定時間檢查（例如每天早上 9:00）
schedule.every().day.at("09:00").do(monitor.check_updates)

# 或者每隔 N 小時檢查一次
# schedule.every(6).hours.do(monitor.check_updates)

# 或者每隔 N 分鐘檢查一次（測試用）
# schedule.every(5).minutes.do(monitor.check_updates)

print("爬蟲已啟動，等待排程執行...")
print("按 Ctrl+C 停止")

# 執行排程（這會一直運行）
# 注意：在 Jupyter Notebook 中運行會阻塞當前 cell
while True:
    schedule.run_pending()
    time.sleep(60)  # 每分鐘檢查一次是否有待執行的任務

## 方法二：使用 Linux Cron Job（推薦用於伺服器）

這是最穩定和最常用的方法。

### 步驟：
1. 將上面的程式碼儲存為 Python 檔案（例如 `webpage_monitor.py`）
2. 設定 cron job：
```bash
# 編輯 crontab
crontab -e

# 加入以下行（每天早上 9:00 執行）
0 9 * * * /usr/bin/python3 /path/to/webpage_monitor.py >> /path/to/monitor.log 2>&1

# 或每 6 小時執行一次
0 */6 * * * /usr/bin/python3 /path/to/webpage_monitor.py >> /path/to/monitor.log 2>&1
```

### Cron 時間格式說明：
```
* * * * * 指令
│ │ │ │ │
│ │ │ │ └─ 星期幾 (0-7, 0 和 7 都代表星期日)
│ │ │ └─── 月份 (1-12)
│ │ └───── 日期 (1-31)
│ └─────── 小時 (0-23)
└───────── 分鐘 (0-59)
```

## 進階功能：加入通知

當網頁更新時發送通知。

In [ ]:
# 選項 1: 使用 Email 通知
import smtplib
from email.mime.text import MIMEText

def send_email_notification(subject, body):
    """發送 Email 通知"""
    sender = "your_email@gmail.com"
    receiver = "receiver@example.com"
    password = "your_app_password"  # Gmail 需要使用應用程式密碼
    
    msg = MIMEText(body)
    msg['Subject'] = subject
    msg['From'] = sender
    msg['To'] = receiver
    
    try:
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
            server.login(sender, password)
            server.send_message(msg)
        print("Email 通知已發送")
    except Exception as e:
        print(f"發送 Email 失敗: {e}")

# 在 WebsiteMonitor 類別的 check_updates 方法中
# 當偵測到更新時，加入：
# send_email_notification(
#     "網頁更新通知",
#     f"課程網頁已更新！\n更新時間：{datetime.now()}\n網址：{self.url}"
# )

In [ ]:
# 選項 2: 使用 Line Notify（台灣常用）
def send_line_notification(message):
    """發送 Line Notify 通知"""
    token = "YOUR_LINE_NOTIFY_TOKEN"  # 從 https://notify-bot.line.me/ 取得
    url = "https://notify-api.line.me/api/notify"
    
    headers = {
        "Authorization": f"Bearer {token}"
    }
    
    data = {
        "message": message
    }
    
    try:
        response = requests.post(url, headers=headers, data=data)
        if response.status_code == 200:
            print("Line 通知已發送")
        else:
            print(f"發送 Line 通知失敗: {response.status_code}")
    except Exception as e:
        print(f"發送 Line 通知時發生錯誤: {e}")

# 使用範例：
# send_line_notification(f"\n⚠️ 課程網頁已更新！\n時間：{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n網址：{URL}")

## 快速開始步驟

1. **替換你的帳號密碼**：在上面的程式碼中修改 `USERNAME` 和 `PASSWORD`
2. **執行第一次檢查**：運行主要程式碼 cell，建立初始哈希值
3. **選擇定期執行方式**：
   - **最快測試**：直接運行含 schedule 的 cell（方法一）
   - **長期使用**：設定 cron job（方法二）
4. **（可選）加入通知功能**：選擇 Email 或 Line Notify

## 注意事項

- 首次執行會儲存網頁的初始狀態
- 哈希值儲存在 `page_hash.json`
- 網頁快照會儲存為 `page_snapshot_YYYYMMDD_HHMMSS.html`
- 如果登入方式是表單登入，需要根據實際的登入頁面調整 `login()` 方法
- 請確認該網站的使用條款允許爬蟲存取

---
## ✨ Docker 部署方式（推薦用於生產環境）

我已經在同目錄建立了完整的 Docker 環境！

### 檔案清單：
- `webpage_monitor.py` - 主程式（從 notebook 提取）
- `Dockerfile` - Docker 映像檔（單次執行）
- `Dockerfile.cron` - Docker 映像檔（含 cron 定期執行）
- `docker-compose.yml` - 單次執行設定
- `docker-compose-cron.yml` - **自動定期執行設定（推薦）**
- `docker-entrypoint.sh` - Cron 容器啟動腳本
- `requirements.txt` - Python 套件清單
- `README.md` - 完整使用說明

### 快速啟動（三步驟）：

1. **修改帳號密碼**：編輯 `docker-compose-cron.yml`
2. **啟動容器**：執行下方 cell 的指令
3. **完成**！容器會自動在背景定期檢查網頁

### 步驟 1：修改帳號密碼

執行下方指令編輯設定檔（或手動用編輯器開啟）：

In [ ]:
# 查看設定檔內容
cat docker-compose-cron.yml

# 或使用編輯器修改
# nano docker-compose-cron.yml

### 步驟 2：建立並啟動 Docker 容器

In [ ]:
%%bash
cd /home/under115b/work/lungyu_workspace/deep_learning/other_works

# 建立並啟動容器（背景執行）
docker-compose -f docker-compose-cron.yml up -d --build

# 查看容器狀態
docker-compose -f docker-compose-cron.yml ps

### 步驟 3：查看執行日誌

In [ ]:
%%bash
cd /home/under115b/work/lungyu_workspace/deep_learning/other_works

# 查看即時日誌（按 Ctrl+C 停止）
docker-compose -f docker-compose-cron.yml logs -f

# 或只查看最後 50 行
# docker-compose -f docker-compose-cron.yml logs --tail=50

### 常用管理指令

In [ ]:
%%bash
cd /home/under115b/work/lungyu_workspace/deep_learning/other_works

# 手動觸發一次檢查（測試用）
docker-compose -f docker-compose-cron.yml exec webpage-monitor-cron python /app/webpage_monitor.py

# 查看儲存的哈希記錄
cat data/page_hash.json

# 列出所有網頁快照
ls -lh data/page_snapshot_*.html 2>/dev/null || echo "尚無快照"

# 停止容器
# docker-compose -f docker-compose-cron.yml down

# 重新啟動容器
# docker-compose -f docker-compose-cron.yml restart

### Cron 排程設定範例

在 `docker-compose-cron.yml` 中修改 `CRON_SCHEDULE` 環境變數：

```yaml
environment:
  - CRON_SCHEDULE=0 9 * * *  # 每天早上 9:00
```

**常用排程：**
- `0 */6 * * *` → 每 6 小時執行一次
- `*/30 * * * *` → 每 30 分鐘執行一次
- `0 9,21 * * *` → 每天 9:00 和 21:00 執行
- `0 9 * * 1-5` → 週一到週五早上 9:00 執行

修改後需重新啟動容器：
```bash
docker-compose -f docker-compose-cron.yml down
docker-compose -f docker-compose-cron.yml up -d
```

### 優點總結

**使用 Docker 的好處：**
✅ **環境隔離**：不會影響系統其他部分  
✅ **自動重啟**：系統重開機後自動恢復運行  
✅ **資料持久化**：資料儲存在 `./data` 目錄  
✅ **易於管理**：一鍵啟動、停止、查看日誌  
✅ **可移植性**：可以輕鬆部署到任何有 Docker 的機器  
✅ **內建 Cron**：不需要設定系統 cron job  

**檔案位置：**
- 所有 Docker 相關檔案都在當前目錄
- 詳細說明請看 `README.md`